# 04D1I_v2 — LANDFIRE Archived-Vintage Full Production

**Build ID: `04D1I_V2_LANDFIRE_ARCHIVED_VINTAGE_FULL_PRODUCTION`**

Scale the **frozen 04D1H local-raster backend** over the complete archived-vintage
workload:

- LF2019L → 2021 fires: 101,517 units / 492 blocks
- LF2020 → 2022 fires: 31,232 units / 198 blocks
- total: 132,749 units / 690 blocks

No feature-definition change is allowed relative to 04D1C V5 / 04D1H.
This notebook adds only resumable production mechanics and final exact coverage/schema QA.

## Frozen archive ingestion contract

- GeoTIFF-declared NoData: `32767`
- embedded archive sentinel: `-9999`
- analysis missing representation: `NaN`

Both source missing forms are removed before aggregation; zero remains valid.

## 0. Configuration

## V2 correction — zero-direction production blocks

If a block contains zero units with reliable approach direction, the frozen
context builder correctly emits only the line region. Directional frozen V5
fields may therefore be structurally absent from the intermediate DataFrame.

V2 permits only those directional fields to be materialized as `NaN`, and only
for an all-zero-direction block. Missing non-directional fields remain a hard
failure. Blocks containing any reliable-direction unit still require ordinary
full V5 emission.

The production root is unchanged so all previously validated shards are reused.


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = (
    "04D1I_V2_LANDFIRE_ARCHIVED_VINTAGE_FULL_PRODUCTION"
)

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

APPROACH_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_extended_exact_approach_v3" /
    "extended_exact_approach_features_v3.parquet"
)

TERRAIN_UNIT_INDEX_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "terrain_operational_unit_index_v1.parquet"
)

TERRAIN_BLOCK_SAMPLE_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "block_samples"
)

FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT / "outputs" /
    "landfire_feature_architecture_v5" /
    "LANDFIRE_FEATURE_DICTIONARY_V5.csv"
)

ARCHIVE_PROVENANCE_MANIFEST_PATH = (
    PROJECT_ROOT / "outputs" /
    "landfire_archive_acquisition_v7" /
    "LANDFIRE_ARCHIVE_PROVENANCE_MANIFEST_VALIDATED_V7.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_archived_vintage_production_v1"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_archived_vintage_production_v1"
)

SHARD_ROOT = (
    PROCESSED_ROOT / "feature_shards"
)

PROGRESS_ROOT = (
    OUTPUT_ROOT / "progress"
)

for p in [
    OUTPUT_ROOT,
    PROCESSED_ROOT,
    SHARD_ROOT,
    PROGRESS_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

for p in [
    APPROACH_PATH,
    TERRAIN_UNIT_INDEX_PATH,
    TERRAIN_BLOCK_SAMPLE_ROOT,
    FEATURE_DICTIONARY_PATH,
    ARCHIVE_PROVENANCE_MANIFEST_PATH,
]:
    assert p.exists(), p

VERSION_BY_FIRE_YEAR = {
    2020: "LF2016R",
    2021: "LF2019L",
    2022: "LF2020",
    2023: "LF2022",
    2024: "LF2023",
}

ARCHIVED_VERSIONS = [
    "LF2019L",
    "LF2020",
]

PRODUCTS = [
    "FBFM40",
    "CC",
    "CH",
    "CBH",
    "CBD",
]

REGION_DEPTHS_M = {
    "line": [0.0],
    "fireward_near": [-30.0, -60.0, -90.0],
    "fireward_far": [-150.0, -210.0, -300.0],
    "protected_near": [30.0, 60.0, 90.0],
    "protected_far": [150.0, 210.0, 300.0],
}

DIRECTIONAL_REGIONS = {
    "fireward_near",
    "fireward_far",
    "protected_near",
    "protected_far",
}

BLOCK_SIZE_M = 10_000.0

EXPECTED_CRS_EPSG = 5070
EXPECTED_PIXEL_SIZE_M = 30.0
EXPECTED_DTYPE = "int16"
EXPECTED_DECLARED_NODATA = 32767.0

ARCHIVE_EMBEDDED_NODATA_SENTINELS = (
    -9999.0,
)

RESUME_VALID_SHARDS = True
PROGRESS_SAVE_EVERY_BLOCKS = 10
GC_EVERY_BLOCKS = 10

EXPECTED_VERSION_UNITS = {
    "LF2019L": 101_517,
    "LF2020": 31_232,
}

EXPECTED_VERSION_BLOCKS = {
    "LF2019L": 492,
    "LF2020": 198,
}

EXPECTED_TOTAL_UNITS = 132_749
EXPECTED_TOTAL_BLOCKS = 690

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("RESUME_VALID_SHARDS:", RESUME_VALID_SHARDS)
print("EXPECTED_TOTAL_BLOCKS:", EXPECTED_TOTAL_BLOCKS)
print("EXPECTED_TOTAL_UNITS:", EXPECTED_TOTAL_UNITS)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "rasterio": "rasterio",
    "pyarrow": "pyarrow",
}

for module, package in required.items():
    if importlib.util.find_spec(
        module
    ) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

from datetime import datetime, timezone
import hashlib
import json
import math

import numpy as np
import pandas as pd
import rasterio

from rasterio.crs import CRS
from rasterio.windows import Window

pd.set_option(
    "display.max_columns",
    500,
)
pd.set_option(
    "display.max_rows",
    500,
)
pd.set_option(
    "display.width",
    380,
)

import gc
import time
import traceback

## 2. Frozen archived-vintage unit universe + V5 feature dictionary

In [ ]:
unit_index = pd.read_parquet(TERRAIN_UNIT_INDEX_PATH)
approach = pd.read_parquet(APPROACH_PATH)
feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)

FROZEN_V5_FEATURE_SET = set(
    feature_dictionary["feature"].astype(str)
)

assert len(FROZEN_V5_FEATURE_SET) == len(feature_dictionary)

assert unit_index["unit_uid"].is_unique
assert approach["unit_uid"].is_unique
assert feature_dictionary["feature"].is_unique

required_unit_cols = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "terrain_block_id",
    "terrain_block_ix",
    "terrain_block_iy",
]

missing = [
    c for c in required_unit_cols
    if c not in unit_index.columns
]
if missing:
    raise RuntimeError(
        f"Missing terrain unit-index columns: {missing}"
    )

approach_cols = [
    "unit_uid",
    "approach_azimuth_deg",
    "approach_direction_reliable",
    "approach_vs_line_angle_deg",
]

missing = [
    c for c in approach_cols
    if c not in approach.columns
]
if missing:
    raise RuntimeError(
        f"Missing 04A columns: {missing}"
    )

base = unit_index.copy()

base["Year"] = pd.to_numeric(
    base["Year"],
    errors="raise",
).astype(int)

base["landfire_version_key"] = (
    base["Year"].map(VERSION_BY_FIRE_YEAR)
)

# terrain_operational_unit_index_v1 may already carry one or more
# 04A direction fields. Drop those copies before merging the canonical
# 04A table; otherwise pandas creates *_x / *_y suffixed columns and the
# unsuffixed field names disappear.
overlap_04a_cols = [
    c
    for c in approach_cols
    if c != "unit_uid" and c in base.columns
]

print("Overlapping 04A columns already in terrain unit index:")
print(overlap_04a_cols)

base = base.drop(
    columns=overlap_04a_cols,
    errors="ignore",
)

base = base.merge(
    approach[approach_cols],
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

# Canonical post-merge schema: no pandas suffix artifacts allowed.
suffix_artifacts = [
    c
    for c in base.columns
    if c.endswith("_x") or c.endswith("_y")
]

if suffix_artifacts:
    raise RuntimeError(
        f"Unexpected merge suffix columns remain: {suffix_artifacts}"
    )

for c in approach_cols:
    if c not in base.columns:
        raise RuntimeError(
            f"Canonical 04A column missing after merge: {c}"
        )

print("04A CANONICAL MERGE: PASS")

base["landfire_directional_context_valid"] = (
    base["approach_direction_reliable"]
    .fillna(False)
    & pd.to_numeric(
        base["approach_azimuth_deg"],
        errors="coerce",
    ).notna()
)

archived_base = base[
    base["landfire_version_key"].isin(
        ARCHIVED_VERSIONS
    )
].copy()

print("Operational units:", f"{len(base):,}")
print("Archived-vintage units:", f"{len(archived_base):,}")
print(
    "Frozen V5 fields:",
    f"{len(feature_dictionary):,}",
)

display(
    feature_dictionary.groupby(
        ["tier", "product"],
        as_index=False,
    ).agg(
        features=("feature", "size")
    )
)

## 3. Validated archive source contract + native-grid phase QA

In [ ]:
archive_manifest = pd.read_csv(
    ARCHIVE_PROVENANCE_MANIFEST_PATH
)

required_manifest_cols = [
    "version_key",
    "product",
    "tif_path",
    "acquisition_validated",
]

missing = [
    c for c in required_manifest_cols
    if c not in archive_manifest.columns
]

if missing:
    raise RuntimeError(
        f"Archive provenance manifest missing columns: {missing}"
    )

archive_sources = archive_manifest[
    archive_manifest["version_key"].isin(
        ARCHIVED_VERSIONS
    )
    & archive_manifest["product"].isin(
        PRODUCTS
    )
].copy()

archive_sources[
    "acquisition_validated_bool"
] = (
    archive_sources[
        "acquisition_validated"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

assert len(archive_sources) == 10
assert not archive_sources[
    ["version_key", "product"]
].duplicated().any()
assert archive_sources[
    "acquisition_validated_bool"
].all()

RASTER_PATHS = {}
source_meta_rows = []

for _, row in archive_sources.iterrows():

    version_key = row["version_key"]
    product = row["product"]
    path = Path(row["tif_path"])

    if not path.exists():
        raise RuntimeError(
            f"Validated archive GeoTIFF is missing: {path}"
        )

    RASTER_PATHS[
        (version_key, product)
    ] = path

    with rasterio.open(path) as src_r:

        info = {
            "version_key": version_key,
            "product": product,
            "tif_path": str(path),
            "width": src_r.width,
            "height": src_r.height,
            "count": src_r.count,
            "dtype": src_r.dtypes[0],
            "crs_epsg": (
                src_r.crs.to_epsg()
                if src_r.crs is not None
                else None
            ),
            "nodata": src_r.nodata,
            "pixel_x_m": abs(src_r.transform.a),
            "pixel_y_m": abs(src_r.transform.e),
            "transform_a": src_r.transform.a,
            "transform_b": src_r.transform.b,
            "transform_c": src_r.transform.c,
            "transform_d": src_r.transform.d,
            "transform_e": src_r.transform.e,
            "transform_f": src_r.transform.f,
        }

    info["count_ok"] = (
        info["count"] == 1
    )

    info["dtype_ok"] = (
        str(info["dtype"]).lower()
        == EXPECTED_DTYPE
    )

    info["crs_ok"] = (
        info["crs_epsg"]
        == EXPECTED_CRS_EPSG
    )

    info["pixel_ok"] = (
        np.isclose(
            info["pixel_x_m"],
            EXPECTED_PIXEL_SIZE_M,
            atol=1e-9,
            rtol=0,
        )
        and np.isclose(
            info["pixel_y_m"],
            EXPECTED_PIXEL_SIZE_M,
            atol=1e-9,
            rtol=0,
        )
    )

    info["north_up_ok"] = (
        np.isclose(
            info["transform_b"],
            0.0,
            atol=1e-12,
            rtol=0,
        )
        and np.isclose(
            info["transform_d"],
            0.0,
            atol=1e-12,
            rtol=0,
        )
    )

    info["declared_nodata_ok"] = (
        info["nodata"] is not None
        and float(info["nodata"])
        == EXPECTED_DECLARED_NODATA
    )

    source_meta_rows.append(info)


source_metadata_qa = pd.DataFrame(
    source_meta_rows
)

metadata_check_cols = [
    "count_ok",
    "dtype_ok",
    "crs_ok",
    "pixel_ok",
    "north_up_ok",
    "declared_nodata_ok",
]

assert source_metadata_qa[
    metadata_check_cols
].all().all()


def phase_residual(
    delta,
    spacing,
):
    return abs(
        delta
        - round(delta / spacing)
        * spacing
    )


grid_alignment_rows = []

for version_key, g in source_metadata_qa.groupby(
    "version_key",
    sort=False,
):

    ref = g.loc[
        g["product"] == "FBFM40"
    ].iloc[0]

    for _, z in g.iterrows():

        dx = (
            float(z["transform_c"])
            - float(ref["transform_c"])
        )

        dy = (
            float(z["transform_f"])
            - float(ref["transform_f"])
        )

        phase_x = phase_residual(
            dx,
            EXPECTED_PIXEL_SIZE_M,
        )

        phase_y = phase_residual(
            dy,
            EXPECTED_PIXEL_SIZE_M,
        )

        same_crs = (
            z["crs_epsg"]
            == ref["crs_epsg"]
        )

        same_pixel_size = (
            np.isclose(
                z["pixel_x_m"],
                ref["pixel_x_m"],
                atol=1e-9,
                rtol=0,
            )
            and np.isclose(
                z["pixel_y_m"],
                ref["pixel_y_m"],
                atol=1e-9,
                rtol=0,
            )
        )

        same_orientation = (
            np.isclose(
                z["transform_a"],
                ref["transform_a"],
                atol=1e-12,
                rtol=0,
            )
            and np.isclose(
                z["transform_e"],
                ref["transform_e"],
                atol=1e-12,
                rtol=0,
            )
            and np.isclose(
                z["transform_b"],
                ref["transform_b"],
                atol=1e-12,
                rtol=0,
            )
            and np.isclose(
                z["transform_d"],
                ref["transform_d"],
                atol=1e-12,
                rtol=0,
            )
        )

        aligned = (
            same_crs
            and same_pixel_size
            and same_orientation
            and phase_x <= 1e-6
            and phase_y <= 1e-6
        )

        grid_alignment_rows.append({
            "version_key": version_key,
            "reference_product": "FBFM40",
            "comparison_product": z["product"],
            "same_crs": same_crs,
            "same_pixel_size": same_pixel_size,
            "same_orientation": same_orientation,
            "origin_dx_m": dx,
            "origin_dy_m": dy,
            "phase_x_residual_m": phase_x,
            "phase_y_residual_m": phase_y,
            "native_grid_phase_aligned": aligned,
        })


source_grid_alignment_qa = pd.DataFrame(
    grid_alignment_rows
)

assert source_grid_alignment_qa[
    "native_grid_phase_aligned"
].all()

display(source_metadata_qa)
display(source_grid_alignment_qa)

source_metadata_qa.to_csv(
    OUTPUT_ROOT /
    "ARCHIVE_SOURCE_METADATA_QA_V1.csv",
    index=False,
)

source_grid_alignment_qa.to_csv(
    OUTPUT_ROOT /
    "ARCHIVE_SOURCE_GRID_ALIGNMENT_QA_V1.csv",
    index=False,
)

print(
    "ARCHIVE LOCAL-RASTER SOURCE CONTRACT: PASS"
)

## 4. Full archived-vintage block workload

In [ ]:
block_workload = (
    archived_base.groupby(
        [
            "landfire_version_key",
            "terrain_block_id",
            "terrain_block_ix",
            "terrain_block_iy",
        ],
        as_index=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        directional_valid_units=(
            "landfire_directional_context_valid",
            "sum",
        ),
    )
)

block_workload = block_workload.sort_values(
    [
        "landfire_version_key",
        "terrain_block_ix",
        "terrain_block_iy",
        "terrain_block_id",
    ]
).reset_index(
    drop=True
)

version_workload = (
    block_workload.groupby(
        "landfire_version_key",
        as_index=False,
    )
    .agg(
        blocks=(
            "terrain_block_id",
            "nunique",
        ),
        units=(
            "units",
            "sum",
        ),
        directional_valid_units=(
            "directional_valid_units",
            "sum",
        ),
    )
)

display(version_workload)

assert len(block_workload) == EXPECTED_TOTAL_BLOCKS
assert int(
    block_workload["units"].sum()
) == EXPECTED_TOTAL_UNITS

for version_key in ARCHIVED_VERSIONS:

    row = version_workload[
        version_workload[
            "landfire_version_key"
        ] == version_key
    ].iloc[0]

    assert int(
        row["blocks"]
    ) == EXPECTED_VERSION_BLOCKS[
        version_key
    ]

    assert int(
        row["units"]
    ) == EXPECTED_VERSION_UNITS[
        version_key
    ]

block_workload.to_csv(
    OUTPUT_ROOT /
    "ARCHIVED_VINTAGE_BLOCK_WORKLOAD_V1.csv",
    index=False,
)

version_workload.to_csv(
    OUTPUT_ROOT /
    "ARCHIVED_VINTAGE_VERSION_WORKLOAD_V1.csv",
    index=False,
)

print(
    "FULL ARCHIVED-VINTAGE WORKLOAD: PASS"
)

## 5. Frozen V5 product semantics

In [ ]:
FBFM40_GROUP_CODES = {
    "NB": {91, 92, 93, 98, 99},
    "GR": set(range(101, 110)),
    "GS": set(range(121, 125)),
    "SH": set(range(141, 150)),
    "TU": set(range(161, 166)),
    "TL": set(range(181, 190)),
    "SB": set(range(201, 205)),
}

FBFM40_VALID_CODES = set().union(
    *FBFM40_GROUP_CODES.values()
)

FBFM40_NONBURNABLE_CODES = set(
    FBFM40_GROUP_CODES["NB"]
)

FBFM40_BURNABLE_CODES = (
    FBFM40_VALID_CODES
    - FBFM40_NONBURNABLE_CODES
)

assert len(FBFM40_VALID_CODES) == 45
assert len(FBFM40_BURNABLE_CODES) == 40
assert len(FBFM40_NONBURNABLE_CODES) == 5

FBFM40_CODE_TO_GROUP = {
    code: group
    for group, codes in FBFM40_GROUP_CODES.items()
    for code in codes
}

FBFM40_NB_SUBCODES = {
    "NB1": 91,
    "NB2": 92,
    "NB3": 93,
    "NB8": 98,
    "NB9": 99,
}

CANOPY_SPEC = {
    "CC": {
        "scale": 1.0,
        "top_censored_raw": None,
    },
    "CH": {
        "scale": 0.1,
        "top_censored_raw": 510,
    },
    "CBH": {
        "scale": 0.1,
        "top_censored_raw": 100,
    },
    "CBD": {
        "scale": 0.01,
        "top_censored_raw": 45,
    },
}

print("V5 PRODUCT SEMANTICS: PASS")

## 6. Frozen weighted aggregation helpers

In [ ]:
def _clean_weighted(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)

    ok = (
        np.isfinite(v)
        & np.isfinite(w)
        & (w > 0)
    )

    return v[ok], w[ok]


def weighted_mean(values, weights):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    return float(
        np.average(v, weights=w)
    )


def weighted_std(values, weights):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    mu = np.average(v, weights=w)

    return float(
        np.sqrt(
            np.average(
                (v - mu) ** 2,
                weights=w,
            )
        )
    )


def weighted_quantile(values, weights, q):
    v, w = _clean_weighted(values, weights)

    if not len(v):
        return np.nan

    order = np.argsort(v)
    v = v[order]
    w = w[order]

    cw = np.cumsum(w)
    target = float(q) * cw[-1]

    idx = np.searchsorted(
        cw,
        target,
        side="left",
    )

    return float(v[idx])


def weighted_fraction(mask, weights):
    mask = np.asarray(mask, dtype=bool)
    w = np.asarray(weights, dtype=float)

    ok = np.isfinite(w) & (w > 0)

    if not ok.any():
        return np.nan

    return float(
        w[ok & mask].sum()
        / w[ok].sum()
    )


def weighted_entropy(categories, weights):
    c = np.asarray(categories)
    w = np.asarray(weights, dtype=float)

    ok = (
        pd.notna(c)
        & np.isfinite(w)
        & (w > 0)
    )

    if not ok.any():
        return np.nan

    c = c[ok]
    w = w[ok]

    totals = {}
    for value, weight in zip(c, w):
        totals[value] = (
            totals.get(value, 0.0)
            + float(weight)
        )

    p = np.asarray(
        list(totals.values()),
        dtype=float,
    )
    p = p / p.sum()

    return float(
        -(p * np.log(p)).sum()
    )


def jensen_shannon_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    if (
        len(p) != len(q)
        or len(p) == 0
        or not np.isfinite(p).all()
        or not np.isfinite(q).all()
        or p.sum() <= 0
        or q.sum() <= 0
    ):
        return np.nan

    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return float(
            np.sum(
                a[mask]
                * np.log(
                    a[mask] / b[mask]
                )
            )
        )

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

## 7. Frozen line-relative context requests

In [ ]:
def find_block_sample_path(block_id):
    direct = (
        TERRAIN_BLOCK_SAMPLE_ROOT /
        f"{block_id}.parquet"
    )

    if direct.exists():
        return direct

    candidates = list(
        TERRAIN_BLOCK_SAMPLE_ROOT.glob(
            f"*{block_id}*.parquet"
        )
    )

    if len(candidates) == 1:
        return candidates[0]

    raise RuntimeError(
        f"Could not resolve block sample shard for {block_id}. "
        f"Candidates={candidates}"
    )


def build_context_requests(
    line_samples,
    unit_direction,
):
    required = {
        "unit_uid",
        "x_m",
        "y_m",
        "sample_weight_m",
    }

    missing = sorted(
        required - set(line_samples.columns)
    )

    if missing:
        raise RuntimeError(
            f"Missing line-sample columns: {missing}"
        )

    x = line_samples.merge(
        unit_direction[
            [
                "unit_uid",
                "approach_azimuth_deg",
                "landfire_directional_context_valid",
            ]
        ],
        on="unit_uid",
        how="left",
        validate="many_to_one",
    ).copy()

    x["line_position_index"] = (
        x.groupby(
            "unit_uid",
            sort=False,
        ).cumcount()
    )

    theta = np.deg2rad(
        pd.to_numeric(
            x["approach_azimuth_deg"],
            errors="coerce",
        ).to_numpy(float)
    )

    east = np.sin(theta)
    north = np.cos(theta)

    frames = []

    line = x[
        [
            "unit_uid",
            "x_m",
            "y_m",
            "sample_weight_m",
            "line_position_index",
        ]
    ].copy()

    line["region"] = "line"
    line["offset_m"] = 0.0
    line["context_weight_m"] = (
        line["sample_weight_m"]
    )

    frames.append(line)

    valid = (
        x["landfire_directional_context_valid"]
        .fillna(False)
        .to_numpy(bool)
    )

    for region in sorted(DIRECTIONAL_REGIONS):
        depths = REGION_DEPTHS_M[region]

        for offset in depths:
            idx = np.flatnonzero(valid)

            g = x.loc[
                valid,
                [
                    "unit_uid",
                    "x_m",
                    "y_m",
                    "sample_weight_m",
                    "line_position_index",
                ],
            ].copy()

            g["x_m"] = (
                g["x_m"].to_numpy(float)
                + float(offset) * east[idx]
            )

            g["y_m"] = (
                g["y_m"].to_numpy(float)
                + float(offset) * north[idx]
            )

            g["region"] = region
            g["offset_m"] = float(offset)

            g["context_weight_m"] = (
                g["sample_weight_m"]
                / len(depths)
            )

            frames.append(g)

    out = pd.concat(
        frames,
        ignore_index=True,
    )

    out["request_id"] = np.arange(
        len(out),
        dtype=np.int64,
    )

    return out

## 8. Validated local archive-raster backend

In [ ]:
def safe_token(
    value,
    n=16,
):
    return hashlib.sha1(
        str(
            value
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :n
    ]


def archive_missing_mask(
    values,
    declared_nodata,
):
    values = np.asarray(
        values
    )

    mask = ~np.isfinite(
        values
    )

    if (
        declared_nodata is not None
        and np.isfinite(
            declared_nodata
        )
    ):
        mask |= (
            values
            == declared_nodata
        )

    for sentinel in (
        ARCHIVE_EMBEDDED_NODATA_SENTINELS
    ):
        mask |= (
            values
            == sentinel
        )

    return mask


def product_domain_ok(
    product,
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(
            values
        )
    ]

    if not len(
        values
    ):
        return True

    if product == "FBFM40":
        return bool(
            np.isin(
                values.astype(
                    np.int64
                ),
                list(
                    FBFM40_VALID_CODES
                ),
            ).all()
        )

    if product == "CC":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 100
                )
            ).all()
        )

    if product == "CH":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 510
                )
            ).all()
        )

    if product == "CBH":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 100
                )
            ).all()
        )

    if product == "CBD":
        return bool(
            (
                (
                    values >= 0
                )
                & (
                    values <= 45
                )
            ).all()
        )

    raise KeyError(
        product
    )


def sample_local_raster_at_requests(
    raster_path,
    requests_df,
    product,
):
    """
    Nearest-neighbor native-cell extraction from one validated full-extent
    archive GeoTIFF.

    Returns normalized product_raw where all source missing representations
    have already been converted to NaN.
    """
    raster_path = Path(
        raster_path
    )

    with rasterio.open(
        raster_path
    ) as src:

        xs = requests_df[
            "x_m"
        ].to_numpy(
            float
        )

        ys = requests_df[
            "y_m"
        ].to_numpy(
            float
        )

        rows, cols = rasterio.transform.rowcol(
            src.transform,
            xs,
            ys,
        )

        rows = np.asarray(
            rows,
            dtype=np.int64,
        )

        cols = np.asarray(
            cols,
            dtype=np.int64,
        )

        inside = (
            (
                rows >= 0
            )
            & (
                rows < src.height
            )
            & (
                cols >= 0
            )
            & (
                cols < src.width
            )
        )

        if not inside.all():
            raise RuntimeError(
                f"{product}: "
                f"{int((~inside).sum()):,} context requests "
                "fell outside the validated full-extent GeoTIFF."
            )

        row_min = int(
            rows.min()
        )
        row_max = int(
            rows.max()
        )
        col_min = int(
            cols.min()
        )
        col_max = int(
            cols.max()
        )

        window = Window(
            col_min,
            row_min,
            col_max
            - col_min
            + 1,
            row_max
            - row_min
            + 1,
        )

        block = src.read(
            1,
            window=window,
        )

        local_rows = (
            rows
            - row_min
        )

        local_cols = (
            cols
            - col_min
        )

        raw_source = block[
            local_rows,
            local_cols,
        ].astype(
            np.float64
        )

        declared_nodata = (
            float(
                src.nodata
            )
            if src.nodata
            is not None
            else None
        )

        missing = archive_missing_mask(
            raw_source,
            declared_nodata,
        )

        raw_normalized = (
            raw_source.copy()
        )

        raw_normalized[
            missing
        ] = np.nan

        center_x, center_y = (
            rasterio.transform.xy(
                src.transform,
                rows,
                cols,
                offset="center",
            )
        )

        center_x = np.asarray(
            center_x,
            dtype=float,
        )

        center_y = np.asarray(
            center_y,
            dtype=float,
        )

        declared_count = (
            int(
                np.sum(
                    raw_source
                    == declared_nodata
                )
            )
            if declared_nodata
            is not None
            else 0
        )

        embedded_counts = {
            str(
                int(
                    sentinel
                )
            ): int(
                np.sum(
                    raw_source
                    == sentinel
                )
            )
            for sentinel in (
                ARCHIVE_EMBEDDED_NODATA_SENTINELS
            )
        }

        valid = raw_normalized[
            np.isfinite(
                raw_normalized
            )
        ]

        qa = {
            "source_path": str(
                raster_path
            ),
            "source_width": src.width,
            "source_height": src.height,
            "source_dtype": src.dtypes[
                0
            ],
            "source_crs_epsg": (
                src.crs.to_epsg()
                if src.crs
                is not None
                else None
            ),
            "source_declared_nodata": (
                declared_nodata
            ),
            "request_count": int(
                len(
                    requests_df
                )
            ),
            "all_requests_inside_source": bool(
                inside.all()
            ),
            "window_row_min": row_min,
            "window_row_max": row_max,
            "window_col_min": col_min,
            "window_col_max": col_max,
            "window_width_px": int(
                window.width
            ),
            "window_height_px": int(
                window.height
            ),
            "window_cells_read": int(
                window.width
                * window.height
            ),
            "missing_request_count": int(
                missing.sum()
            ),
            "declared_nodata_request_count": (
                declared_count
            ),
            "embedded_sentinel_request_counts": (
                json.dumps(
                    embedded_counts,
                    sort_keys=True,
                )
            ),
            "valid_request_count": int(
                np.isfinite(
                    raw_normalized
                ).sum()
            ),
            "sample_min_after_missing_mask": (
                float(
                    np.min(
                        valid
                    )
                )
                if len(
                    valid
                )
                else np.nan
            ),
            "sample_max_after_missing_mask": (
                float(
                    np.max(
                        valid
                    )
                )
                if len(
                    valid
                )
                else np.nan
            ),
            "domain_ok": product_domain_ok(
                product,
                raw_normalized,
            ),
        }

    if not qa[
        "domain_ok"
    ]:
        raise RuntimeError(
            f"{product}: sampled raw-domain QA failed."
        )

    return (
        pd.DataFrame({
            "request_id": requests_df[
                "request_id"
            ].to_numpy(
                np.int64
            ),
            f"{product}_row": rows,
            f"{product}_col": cols,
            f"{product}_cell_center_x_m": (
                center_x
            ),
            f"{product}_cell_center_y_m": (
                center_y
            ),
            f"{product}_raw": (
                raw_normalized
            ),
        }),
        qa,
    )


print(
    "LOCAL ARCHIVE RASTER BACKEND: READY"
)

## 9. Duplicate native-cell collapse

In [ ]:
def collapse_product_cells(
    sampled_requests,
    product,
):
    row_col = [
        f"{product}_row",
        f"{product}_col",
    ]

    required = [
        "unit_uid",
        "region",
        "context_weight_m",
        f"{product}_raw",
        *row_col,
    ]

    missing = [
        c for c in required
        if c not in sampled_requests.columns
    ]

    if missing:
        raise RuntimeError(
            f"Missing columns for {product}: {missing}"
        )

    value_check = (
        sampled_requests.groupby(
            [
                "unit_uid",
                "region",
                *row_col,
            ],
            sort=False,
        )[f"{product}_raw"]
        .nunique(dropna=False)
    )

    if (value_check > 1).any():
        raise RuntimeError(
            f"{product}: one native cell produced multiple raw values."
        )

    out = (
        sampled_requests.groupby(
            [
                "unit_uid",
                "region",
                *row_col,
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            raw_value=(f"{product}_raw", "first"),
            context_weight_m=("context_weight_m", "sum"),
            contributing_requests=("request_id", "size"),
        )
    )

    return out

## 10. FBFM40 + canopy aggregation

In [ ]:
def aggregate_fbfm40_group(g):
    raw = pd.to_numeric(
        g["raw_value"],
        errors="coerce",
    ).to_numpy(float)

    w = g[
        "context_weight_m"
    ].to_numpy(float)

    valid = (
        np.isfinite(raw)
        & np.isin(
            raw.astype(
                int,
                copy=False,
            ),
            list(FBFM40_VALID_CODES),
        )
    )

    out = {
        "valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
        "unique_cell_count": int(
            len(g)
        ),
    }

    if not valid.any():
        for group in FBFM40_GROUP_CODES:
            out[f"group_{group}_fraction"] = np.nan

        for nb_name in FBFM40_NB_SUBCODES:
            out[f"{nb_name.lower()}_fraction"] = np.nan

        out["group_entropy"] = np.nan
        out["exact_entropy"] = np.nan
        out["unique_exact_models"] = np.nan
        out["dominant_group_fraction"] = np.nan

        for code_value in sorted(
            FBFM40_VALID_CODES
        ):
            out[
                f"exact_code_{code_value}_fraction"
            ] = np.nan

        return out

    vr = raw[valid].astype(int)
    vw = w[valid]

    groups = np.asarray([
        FBFM40_CODE_TO_GROUP[int(code)]
        for code in vr
    ])

    group_fracs = []

    for group in FBFM40_GROUP_CODES:
        frac = weighted_fraction(
            groups == group,
            vw,
        )

        out[
            f"group_{group}_fraction"
        ] = frac

        group_fracs.append(frac)

    for nb_name, nb_code in (
        FBFM40_NB_SUBCODES.items()
    ):
        out[
            f"{nb_name.lower()}_fraction"
        ] = weighted_fraction(
            vr == nb_code,
            vw,
        )

    out["group_entropy"] = weighted_entropy(
        groups,
        vw,
    )

    out["exact_entropy"] = weighted_entropy(
        vr,
        vw,
    )

    out["unique_exact_models"] = int(
        len(np.unique(vr))
    )

    out["dominant_group_fraction"] = float(
        np.nanmax(group_fracs)
    )

    for code_value in sorted(
        FBFM40_VALID_CODES
    ):
        out[
            f"exact_code_{code_value}_fraction"
        ] = weighted_fraction(
            vr == code_value,
            vw,
        )

    return out


def aggregate_canopy_group(
    g,
    product,
):
    spec = CANOPY_SPEC[product]

    raw = pd.to_numeric(
        g["raw_value"],
        errors="coerce",
    ).to_numpy(
        dtype=float,
        copy=True,
    )

    w = g[
        "context_weight_m"
    ].to_numpy(float)

    valid = (
        np.isfinite(raw)
    )

    out = {
        "valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
        "unique_cell_count": int(
            len(g)
        ),
    }

    stats = [
        "mean",
        "std",
        "p10",
        "median",
        "p90",
        "zero_fraction",
        "positive_mean",
        "positive_std",
        "positive_p10",
        "positive_median",
        "positive_p90",
        "top_censored_fraction",
    ]

    if not valid.any():
        for stat in stats:
            out[stat] = np.nan
        return out

    vr = raw[valid]
    vw = w[valid]

    physical = (
        vr
        * float(spec["scale"])
    )

    out["mean"] = weighted_mean(
        physical,
        vw,
    )

    out["std"] = weighted_std(
        physical,
        vw,
    )

    out["p10"] = weighted_quantile(
        physical,
        vw,
        0.10,
    )

    out["median"] = weighted_quantile(
        physical,
        vw,
        0.50,
    )

    out["p90"] = weighted_quantile(
        physical,
        vw,
        0.90,
    )

    out["zero_fraction"] = weighted_fraction(
        vr == 0,
        vw,
    )

    positive = vr > 0

    if positive.any():
        vp = physical[positive]
        wp = vw[positive]

        out["positive_mean"] = weighted_mean(vp, wp)
        out["positive_std"] = weighted_std(vp, wp)
        out["positive_p10"] = weighted_quantile(vp, wp, 0.10)
        out["positive_median"] = weighted_quantile(vp, wp, 0.50)
        out["positive_p90"] = weighted_quantile(vp, wp, 0.90)
    else:
        out["positive_mean"] = np.nan
        out["positive_std"] = np.nan
        out["positive_p10"] = np.nan
        out["positive_median"] = np.nan
        out["positive_p90"] = np.nan

    top = spec[
        "top_censored_raw"
    ]

    out["top_censored_fraction"] = (
        weighted_fraction(
            vr == top,
            vw,
        )
        if top is not None
        else np.nan
    )

    return out

## 11. Joint canopy-derived features

In [ ]:
def build_joint_canopy_cells(
    sampled_requests,
    center_tolerance_m=1e-6,
):
    """
    Build joint CH/CBH/CBD native-cell table.

    Critical contract:
      the three products must select the same PHYSICAL native-cell center
      for every request. Local tile row/col equality is not required.
    """

    required = []

    for product in ["CH", "CBH", "CBD"]:
        required.extend([
            f"{product}_cell_center_x_m",
            f"{product}_cell_center_y_m",
            f"{product}_raw",
        ])

    missing = [
        c for c in required
        if c not in sampled_requests.columns
    ]

    if missing:
        raise RuntimeError(
            f"Missing joint-canopy columns: {missing}"
        )

    ch_x = sampled_requests[
        "CH_cell_center_x_m"
    ].to_numpy(float)

    ch_y = sampled_requests[
        "CH_cell_center_y_m"
    ].to_numpy(float)

    alignment_rows = []

    for product in ["CBH", "CBD"]:
        px = sampled_requests[
            f"{product}_cell_center_x_m"
        ].to_numpy(float)

        py = sampled_requests[
            f"{product}_cell_center_y_m"
        ].to_numpy(float)

        dx = np.abs(px - ch_x)
        dy = np.abs(py - ch_y)

        max_dx = float(
            np.nanmax(dx)
        )

        max_dy = float(
            np.nanmax(dy)
        )

        alignment_rows.append({
            "reference_product": "CH",
            "comparison_product": product,
            "max_abs_center_dx_m": max_dx,
            "max_abs_center_dy_m": max_dy,
            "all_centers_aligned": bool(
                (dx <= center_tolerance_m).all()
                and (dy <= center_tolerance_m).all()
            ),
        })

    alignment = pd.DataFrame(
        alignment_rows
    )

    if not alignment[
        "all_centers_aligned"
    ].all():
        raise RuntimeError(
            "CH/CBH/CBD native cell centers are not aligned.\\n"
            + alignment.to_string(index=False)
        )

    x = sampled_requests[
        [
            "unit_uid",
            "region",
            "context_weight_m",
            "CH_cell_center_x_m",
            "CH_cell_center_y_m",
            "CH_raw",
            "CBH_raw",
            "CBD_raw",
        ]
    ].copy()

    # Use CH physical center as canonical joint-canopy cell identity because
    # alignment to CBH/CBD was asserted above.
    x = x.rename(
        columns={
            "CH_cell_center_x_m": "cell_center_x_m",
            "CH_cell_center_y_m": "cell_center_y_m",
        }
    )

    value_cols = [
        "CH_raw",
        "CBH_raw",
        "CBD_raw",
    ]

    for value_col in value_cols:
        n = (
            x.groupby(
                [
                    "unit_uid",
                    "region",
                    "cell_center_x_m",
                    "cell_center_y_m",
                ],
                sort=False,
            )[value_col]
            .nunique(dropna=False)
        )

        if (n > 1).any():
            raise RuntimeError(
                f"Multiple {value_col} values at one physical native cell."
            )

    cells = (
        x.groupby(
            [
                "unit_uid",
                "region",
                "cell_center_x_m",
                "cell_center_y_m",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            context_weight_m=(
                "context_weight_m",
                "sum",
            ),
            CH_raw=("CH_raw", "first"),
            CBH_raw=("CBH_raw", "first"),
            CBD_raw=("CBD_raw", "first"),
        )
    )

    ch = cells["CH_raw"].to_numpy(float)
    cbh = cells["CBH_raw"].to_numpy(float)
    cbd = cells["CBD_raw"].to_numpy(float)

    joint_valid = (
        np.isfinite(ch)
        & np.isfinite(cbh)
        & np.isfinite(cbd)
    )

    ch_m = np.full(
        len(cells),
        np.nan,
    )

    cbh_m = np.full(
        len(cells),
        np.nan,
    )

    cbd_phys = np.full(
        len(cells),
        np.nan,
    )

    ch_m[joint_valid] = (
        ch[joint_valid] / 10.0
    )

    cbh_m[joint_valid] = (
        cbh[joint_valid] / 10.0
    )

    cbd_phys[joint_valid] = (
        cbd[joint_valid] / 100.0
    )

    inconsistent = (
        joint_valid
        & (cbh_m > ch_m)
    )

    depth = np.full(
        len(cells),
        np.nan,
    )

    depth[joint_valid] = np.maximum(
        ch_m[joint_valid]
        - cbh_m[joint_valid],
        0.0,
    )

    cfl = np.full(
        len(cells),
        np.nan,
    )

    cfl[joint_valid] = (
        cbd_phys[joint_valid]
        * depth[joint_valid]
    )

    cells["joint_valid"] = joint_valid
    cells["cbh_gt_ch"] = inconsistent
    cells["crown_depth_m"] = depth
    cells["cfl_kg_m2"] = cfl

    return cells, alignment

def aggregate_joint_canopy_group(g):
    w = g[
        "context_weight_m"
    ].to_numpy(float)

    valid = g[
        "joint_valid"
    ].to_numpy(bool)

    out = {
        "joint_valid_weight_fraction": weighted_fraction(
            valid,
            w,
        ),
        "cbh_gt_ch_fraction": weighted_fraction(
            g["cbh_gt_ch"].to_numpy(bool),
            w,
        ),
    }

    if not valid.any():
        out.update({
            "crown_depth_mean": np.nan,
            "crown_depth_p90": np.nan,
            "cfl_mean": np.nan,
            "cfl_p90": np.nan,
        })
        return out

    vw = w[valid]

    depth = g.loc[
        valid,
        "crown_depth_m",
    ].to_numpy(float)

    cfl = g.loc[
        valid,
        "cfl_kg_m2",
    ].to_numpy(float)

    out["crown_depth_mean"] = weighted_mean(
        depth,
        vw,
    )
    out["crown_depth_p90"] = weighted_quantile(
        depth,
        vw,
        0.90,
    )
    out["cfl_mean"] = weighted_mean(
        cfl,
        vw,
    )
    out["cfl_p90"] = weighted_quantile(
        cfl,
        vw,
        0.90,
    )

    return out

## 12. Frozen V5 feature-name emission

In [ ]:
def _emit_if_frozen(out, name, value):
    """
    Emit only exact fields frozen by 04D1C V5.

    Aggregation helpers are allowed to compute richer intermediate statistics,
    but production output must match the frozen feature dictionary exactly.
    """
    if name in FROZEN_V5_FEATURE_SET:
        out[name] = value


def prefix_region_stats(
    unit_uid,
    region,
    product,
    stats,
):
    out = {"unit_uid": unit_uid}

    if product == "FBFM40":
        for key, value in stats.items():

            if key == "valid_weight_fraction":
                name = (
                    f"landfire_{region}_fbfm40_"
                    "valid_weight_fraction"
                )

            elif key == "unique_cell_count":
                name = (
                    f"landfire_{region}_fbfm40_"
                    "unique_cell_count"
                )

            # IMPORTANT: group_entropy is a statistic, not a group fraction.
            elif key == "group_entropy":
                name = (
                    f"landfire_{region}_"
                    "fbfm_group_entropy"
                )

            elif (
                key.startswith("group_")
                and key.endswith("_fraction")
            ):
                suffix = (
                    key[len("group_"):]
                    .removesuffix("_fraction")
                )

                name = (
                    f"landfire_{region}_fbfm_group_"
                    f"{suffix.lower()}_fraction"
                )

            elif key.startswith("nb"):
                name = (
                    f"landfire_{region}_fbfm_{key}"
                )

            elif key.startswith("exact_code_"):
                name = (
                    f"landfire_{region}_fbfm_"
                    f"{key.replace('exact_', '')}"
                )

            else:
                name = (
                    f"landfire_{region}_fbfm_{key}"
                )

            _emit_if_frozen(
                out,
                name,
                value,
            )

    elif product in CANOPY_SPEC:
        for key, value in stats.items():
            name = (
                f"landfire_{region}_"
                f"{product.lower()}_{key}"
            )

            _emit_if_frozen(
                out,
                name,
                value,
            )

    elif product == "DERIVED_CANOPY":
        for key, value in stats.items():

            if key == "joint_valid_weight_fraction":
                name = (
                    f"landfire_{region}_"
                    "derived_canopy_joint_valid_weight_fraction"
                )
            else:
                name = (
                    f"landfire_{region}_{key}"
                )

            _emit_if_frozen(
                out,
                name,
                value,
            )

    else:
        raise KeyError(product)

    return out


# ------------------------------------------------------------------
# Static naming QA for the exact V3 failure mode.
# ------------------------------------------------------------------
_test_stats = {
    "group_entropy": 0.5,
    "group_GR_fraction": 0.7,
    "valid_weight_fraction": 1.0,
    "unique_cell_count": 10,
}

_test_named = prefix_region_stats(
    "qa_unit",
    "line",
    "FBFM40",
    _test_stats,
)

assert (
    "landfire_line_fbfm_group_entropy"
    in _test_named
)

assert (
    "landfire_line_fbfm_group_entropy_fraction"
    not in _test_named
)

# Internal canopy statistics not frozen in V5 must not leak into output.
_canopy_test = prefix_region_stats(
    "qa_unit",
    "line",
    "CBD",
    {
        "mean": 0.1,
        "std": 0.2,       # internal, not V5 for CBD
        "median": 0.1,    # internal, not V5 for CBD
        "positive_mean": 0.2,
    },
)

assert "landfire_line_cbd_mean" in _canopy_test
assert "landfire_line_cbd_positive_mean" in _canopy_test
assert "landfire_line_cbd_std" not in _canopy_test
assert "landfire_line_cbd_median" not in _canopy_test

print("FROZEN-V5 FEATURE-NAME EMISSION CONTRACT: PASS")

## 13. Region feature aggregation

In [ ]:
def aggregate_region_features(
    sampled_requests,
):
    unit_rows = {}

    def update(row):
        uid = row.pop("unit_uid")
        unit_rows.setdefault(
            uid,
            {"unit_uid": uid},
        ).update(row)

    # FBFM40.
    collapsed = collapse_product_cells(
        sampled_requests,
        "FBFM40",
    )

    for (uid, region), g in collapsed.groupby(
        ["unit_uid", "region"],
        sort=False,
    ):
        stats = aggregate_fbfm40_group(g)

        update(
            prefix_region_stats(
                uid,
                region,
                "FBFM40",
                stats,
            )
        )

    # Canopy products.
    for product in [
        "CC",
        "CH",
        "CBH",
        "CBD",
    ]:
        collapsed = collapse_product_cells(
            sampled_requests,
            product,
        )

        for (uid, region), g in collapsed.groupby(
            ["unit_uid", "region"],
            sort=False,
        ):
            stats = aggregate_canopy_group(
                g,
                product,
            )

            update(
                prefix_region_stats(
                    uid,
                    region,
                    product,
                    stats,
                )
            )

    # Joint canopy: requires CH/CBH/CBD physical cell-center alignment.
    joint, canopy_alignment_qa = build_joint_canopy_cells(
        sampled_requests,
    )

    for (uid, region), g in joint.groupby(
        ["unit_uid", "region"],
        sort=False,
    ):
        stats = aggregate_joint_canopy_group(g)

        update(
            prefix_region_stats(
                uid,
                region,
                "DERIVED_CANOPY",
                stats,
            )
        )

    return (
        pd.DataFrame(
            unit_rows.values()
        ),
        canopy_alignment_qa,
    )

## 14. Fireward/protected contrasts

In [ ]:
FBFM_GROUP_ORDER = list(
    FBFM40_GROUP_CODES.keys()
)


def add_contrast_features(features):
    out = features.copy()

    for scale in ["near", "far"]:
        f = f"fireward_{scale}"
        p = f"protected_{scale}"

        # JSD broad-group composition.
        js_values = []

        for _, row in out.iterrows():
            fv = np.asarray([
                row.get(
                    f"landfire_{f}_fbfm_group_{g.lower()}_fraction",
                    np.nan,
                )
                for g in FBFM_GROUP_ORDER
            ], dtype=float)

            pv = np.asarray([
                row.get(
                    f"landfire_{p}_fbfm_group_{g.lower()}_fraction",
                    np.nan,
                )
                for g in FBFM_GROUP_ORDER
            ], dtype=float)

            js_values.append(
                jensen_shannon_divergence(
                    fv,
                    pv,
                )
            )

        out[
            f"landfire_{scale}_fbfm_group_js_divergence"
        ] = js_values

        pairs = [
            (
                "cc_positive_mean",
                f"landfire_{f}_cc_positive_mean",
                f"landfire_{p}_cc_positive_mean",
            ),
            (
                "cbd_positive_mean",
                f"landfire_{f}_cbd_positive_mean",
                f"landfire_{p}_cbd_positive_mean",
            ),
            (
                "crown_depth_mean",
                f"landfire_{f}_crown_depth_mean",
                f"landfire_{p}_crown_depth_mean",
            ),
            (
                "cfl_mean",
                f"landfire_{f}_cfl_mean",
                f"landfire_{p}_cfl_mean",
            ),
            (
                "fbfm_nb8_fraction",
                f"landfire_{f}_fbfm_nb8_fraction",
                f"landfire_{p}_fbfm_nb8_fraction",
            ),
            (
                "fbfm_nb9_fraction",
                f"landfire_{f}_fbfm_nb9_fraction",
                f"landfire_{p}_fbfm_nb9_fraction",
            ),
        ]

        for quantity, fc, pc in pairs:
            out[
                f"landfire_{scale}_fireward_minus_protected_{quantity}"
            ] = (
                pd.to_numeric(
                    out.get(fc),
                    errors="coerce",
                )
                - pd.to_numeric(
                    out.get(pc),
                    errors="coerce",
                )
            )

    return out

## 15. Positional features

In [ ]:
def add_positional_features(
    features,
    sampled_requests,
):
    out = features.copy()

    # Build per-request physical CBD and derived CFL.
    x = sampled_requests.copy()

    cbd_raw = pd.to_numeric(
        x["CBD_raw"],
        errors="coerce",
    ).to_numpy(float)

    ch_raw = pd.to_numeric(
        x["CH_raw"],
        errors="coerce",
    ).to_numpy(float)

    cbh_raw = pd.to_numeric(
        x["CBH_raw"],
        errors="coerce",
    ).to_numpy(float)

    cbd_valid = (
        np.isfinite(cbd_raw)
    )

    joint_valid = (
        cbd_valid
        & np.isfinite(ch_raw)
        & np.isfinite(cbh_raw)
    )

    x["CBD_phys"] = np.nan
    x.loc[
        cbd_valid,
        "CBD_phys",
    ] = (
        cbd_raw[cbd_valid] / 100.0
    )

    x["CFL_phys"] = np.nan

    depth = np.maximum(
        ch_raw[joint_valid] / 10.0
        - cbh_raw[joint_valid] / 10.0,
        0.0,
    )

    x.loc[
        joint_valid,
        "CFL_phys",
    ] = (
        cbd_raw[joint_valid] / 100.0
        * depth
    )

    positional_rows = []

    for region in [
        "fireward_near",
        "fireward_far",
    ]:
        g = x[
            x["region"] == region
        ].copy()

        # One quantity per originating along-line position.
        per_pos_rows = []

        for (
            uid,
            pos,
        ), z in g.groupby(
            [
                "unit_uid",
                "line_position_index",
            ],
            sort=False,
        ):
            # The original line position carries one arc-length weight.
            pos_weight = float(
                z["sample_weight_m"].iloc[0]
            )

            cbd_values = z[
                "CBD_phys"
            ].to_numpy(float)

            cfl_values = z[
                "CFL_phys"
            ].to_numpy(float)

            cbd_pos = cbd_values[
                np.isfinite(cbd_values)
                & (cbd_values > 0)
            ]

            cfl_pos = cfl_values[
                np.isfinite(cfl_values)
            ]

            per_pos_rows.append({
                "unit_uid": uid,
                "line_position_index": pos,
                "position_weight_m": pos_weight,
                "cbd_positive_mean": (
                    float(np.mean(cbd_pos))
                    if len(cbd_pos)
                    else np.nan
                ),
                "cfl_mean": (
                    float(np.mean(cfl_pos))
                    if len(cfl_pos)
                    else np.nan
                ),
            })

        # A block may contain no reliable directional context at all.
        # In that case this region legitimately has no positional rows.
        if not per_pos_rows:
            continue

        per_pos = pd.DataFrame(
            per_pos_rows
        )

        scale = (
            "near"
            if region.endswith("near")
            else "far"
        )

        for uid, z in per_pos.groupby(
            "unit_uid",
            sort=False,
        ):
            row = {"unit_uid": uid}

            for quantity in [
                "cbd_positive_mean",
                "cfl_mean",
            ]:
                vals = z[
                    quantity
                ].to_numpy(float)

                weights = z[
                    "position_weight_m"
                ].to_numpy(float)

                good = (
                    np.isfinite(vals)
                    & np.isfinite(weights)
                    & (weights > 0)
                )

                if good.any():
                    p90 = weighted_quantile(
                        vals[good],
                        weights[good],
                        0.90,
                    )
                    vmax = float(
                        np.max(vals[good])
                    )
                else:
                    p90 = np.nan
                    vmax = np.nan

                row[
                    f"landfire_fireward_{scale}_position_p90_{quantity}"
                ] = p90

                row[
                    f"landfire_fireward_{scale}_position_max_{quantity}"
                ] = vmax

            positional_rows.append(row)

    positional = pd.DataFrame(
        positional_rows
    )

    if positional.empty:
        return out

    positional = (
        positional.groupby(
            "unit_uid",
            as_index=False,
        )
        .first()
    )

    return out.merge(
        positional,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

# Synthetic QA for the exact V1 failure mode: no directional rows.
_positional_zero_direction_features = pd.DataFrame({
    "unit_uid": ["u0"],
    "landfire_line_cbd_mean": [0.0],
})

_positional_zero_direction_sampled = pd.DataFrame({
    "unit_uid": ["u0"],
    "region": ["line"],
    "line_position_index": [0],
    "sample_weight_m": [25.0],
    "CBD_raw": [0],
    "CH_raw": [0],
    "CBH_raw": [0],
})

_positional_zero_direction_out = add_positional_features(
    _positional_zero_direction_features,
    _positional_zero_direction_sampled,
)

assert len(_positional_zero_direction_out) == 1
assert "unit_uid" in _positional_zero_direction_out.columns

print("ZERO-DIRECTION POSITIONAL CONTRACT: PASS")

## 16. Exact production shard schema + resume validation

In [ ]:
V5_FEATURES = list(
    feature_dictionary[
        "feature"
    ].astype(str)
)

V5_FEATURE_SET = set(
    V5_FEATURES
)

DIRECTIONAL_V5_FEATURES = [
    c
    for c in V5_FEATURES
    if (
        "fireward_" in c
        or "protected_" in c
        or "_near_" in c
        or "_far_" in c
    )
]

DIRECTIONAL_V5_FEATURE_SET = set(
    DIRECTIONAL_V5_FEATURES
)

assert DIRECTIONAL_V5_FEATURE_SET.issubset(
    V5_FEATURE_SET
)

print(
    "Frozen directional V5 fields:",
    len(
        DIRECTIONAL_V5_FEATURES
    ),
)

PROVENANCE_META_ORDER = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "terrain_block_id",
    "landfire_version_key",
    "landfire_directional_context_valid",
    "approach_vs_line_angle_deg",
    "landfire_source_status",
]

# Some provenance/QA fields are themselves frozen V5 features.
# Keep only true non-feature metadata outside the V5 feature list.
NON_FEATURE_META = [
    c
    for c in PROVENANCE_META_ORDER
    if c not in V5_FEATURE_SET
]

EXPECTED_SHARD_COLUMNS = (
    NON_FEATURE_META
    + V5_FEATURES
)

assert len(
    EXPECTED_SHARD_COLUMNS
) == len(
    set(
        EXPECTED_SHARD_COLUMNS
    )
)

# Pilot produced 612 total columns; enforce same production width.
assert len(
    EXPECTED_SHARD_COLUMNS
) == 612, (
    len(
        EXPECTED_SHARD_COLUMNS
    ),
    NON_FEATURE_META,
)

print(
    "NON_FEATURE_META:",
    NON_FEATURE_META,
)

print(
    "V5 features:",
    len(
        V5_FEATURES
    ),
)

print(
    "Exact production shard columns:",
    len(
        EXPECTED_SHARD_COLUMNS
    ),
)


def shard_path_for(
    version_key,
    block_id,
):
    return (
        SHARD_ROOT /
        f"{version_key}__"
        f"{safe_token(block_id)}.parquet"
    )


def validate_existing_shard(
    shard_path,
    expected_units,
):
    shard_path = Path(
        shard_path
    )

    if not shard_path.exists():
        return {
            "valid": False,
            "reason": "missing",
            "rows": 0,
            "columns": 0,
        }

    try:
        import pyarrow.parquet as pq

        meta = pq.read_metadata(
            shard_path
        )

        rows = int(
            meta.num_rows
        )

        names = list(
            meta.schema.names
        )

        row_ok = (
            rows
            == int(
                expected_units
            )
        )

        schema_ok = (
            names
            == EXPECTED_SHARD_COLUMNS
        )

        if not (
            row_ok
            and schema_ok
        ):
            return {
                "valid": False,
                "reason": "row_or_schema_mismatch",
                "rows": rows,
                "columns": len(
                    names
                ),
            }

        uid = pd.read_parquet(
            shard_path,
            columns=[
                "unit_uid",
            ],
        )

        unique_ok = (
            len(uid)
            == uid[
                "unit_uid"
            ].nunique()
        )

        if not unique_ok:
            return {
                "valid": False,
                "reason": "duplicate_unit_uid",
                "rows": rows,
                "columns": len(
                    names
                ),
            }

        return {
            "valid": True,
            "reason": "valid",
            "rows": rows,
            "columns": len(
                names
            ),
        }

    except Exception as exc:
        return {
            "valid": False,
            "reason": (
                f"validation_error: "
                f"{type(exc).__name__}: {exc}"
            ),
            "rows": np.nan,
            "columns": np.nan,
        }

## 17. One-block production function

In [ ]:
def run_production_block(
    row,
):
    version_key = row[
        "landfire_version_key"
    ]

    block_id = row[
        "terrain_block_id"
    ]

    expected_units = int(
        row[
            "units"
        ]
    )

    block_units = archived_base[
        (
            archived_base[
                "landfire_version_key"
            ] == version_key
        )
        & (
            archived_base[
                "terrain_block_id"
            ] == block_id
        )
    ].copy()

    if len(
        block_units
    ) != expected_units:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"unit count mismatch"
        )

    sample_path = find_block_sample_path(
        block_id
    )

    line_samples = pd.read_parquet(
        sample_path
    )

    line_samples = line_samples[
        line_samples[
            "unit_uid"
        ].isin(
            block_units[
                "unit_uid"
            ]
        )
    ].copy()

    if line_samples.empty:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            "no line samples"
        )

    requests_df = build_context_requests(
        line_samples,
        block_units[
            [
                "unit_uid",
                "approach_azimuth_deg",
                "landfire_directional_context_valid",
            ]
        ],
    )

    sampled = requests_df.copy()

    for product in PRODUCTS:

        s, qa = sample_local_raster_at_requests(
            RASTER_PATHS[
                (
                    version_key,
                    product,
                )
            ],
            requests_df,
            product,
        )

        if not (
            qa[
                "all_requests_inside_source"
            ]
            and qa[
                "domain_ok"
            ]
        ):
            raise RuntimeError(
                f"{version_key}/{block_id}/{product}: "
                "local raster QA failed"
            )

        sampled = sampled.merge(
            s,
            on="request_id",
            how="left",
            validate="one_to_one",
        )

    # Strong empirical native-cell-center alignment across all products.
    ref_x = sampled[
        "FBFM40_cell_center_x_m"
    ].to_numpy(
        float
    )

    ref_y = sampled[
        "FBFM40_cell_center_y_m"
    ].to_numpy(
        float
    )

    for product in PRODUCTS[
        1:
    ]:

        dx = np.abs(
            sampled[
                f"{product}_cell_center_x_m"
            ].to_numpy(
                float
            )
            - ref_x
        )

        dy = np.abs(
            sampled[
                f"{product}_cell_center_y_m"
            ].to_numpy(
                float
            )
            - ref_y
        )

        if not (
            (
                dx <= 1e-6
            ).all()
            and (
                dy <= 1e-6
            ).all()
        ):
            raise RuntimeError(
                f"{version_key}/{block_id}/{product}: "
                "native cell-center alignment failed"
            )

    features, canopy_alignment_qa = (
        aggregate_region_features(
            sampled
        )
    )

    if (
        len(
            canopy_alignment_qa
        )
        and not canopy_alignment_qa[
            "all_centers_aligned"
        ].all()
    ):
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            "CH/CBH/CBD center alignment failed"
        )

    features = add_contrast_features(
        features
    )

    features = add_positional_features(
        features,
        sampled,
    )

    meta_cols = [
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "_engagement_norm",
        "terrain_block_id",
        "landfire_version_key",
        "landfire_directional_context_valid",
        "approach_vs_line_angle_deg",
    ]

    features = block_units[
        meta_cols
    ].merge(
        features,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    features[
        "landfire_source_status"
    ] = "archive_validated"

    reliable_direction_mask = (
        block_units[
            "landfire_directional_context_valid"
        ].fillna(
            False
        )
    )

    directional_valid_units = int(
        reliable_direction_mask.sum()
    )

    zero_direction_block = (
        directional_valid_units == 0
    )

    produced = set(
        features.columns
    )

    missing_v5_initial = sorted(
        V5_FEATURE_SET
        - produced
    )

    structurally_filled_directional = []

    if missing_v5_initial:

        if not zero_direction_block:
            raise RuntimeError(
                f"{version_key}/{block_id}: "
                "missing frozen V5 features despite having "
                f"{directional_valid_units} reliable-direction unit(s): "
                f"{missing_v5_initial[:20]}"
            )

        missing_non_directional = sorted(
            set(
                missing_v5_initial
            )
            - DIRECTIONAL_V5_FEATURE_SET
        )

        if missing_non_directional:
            raise RuntimeError(
                f"{version_key}/{block_id}: "
                "zero-direction block is missing non-directional V5 "
                f"features: {missing_non_directional[:20]}"
            )

        structurally_filled_directional = list(
            missing_v5_initial
        )

        for feature in structurally_filled_directional:
            features[
                feature
            ] = np.nan

        print(
            f"ZERO-DIRECTION STRUCTURAL FILL: "
            f"{version_key} {block_id} | "
            f"{len(structurally_filled_directional)} "
            "directional V5 fields -> NaN"
        )

    produced = set(
        features.columns
    )

    missing_v5 = sorted(
        V5_FEATURE_SET
        - produced
    )

    if missing_v5:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            "V5 schema still incomplete after structural completion: "
            f"{missing_v5[:20]}"
        )

    unexpected_landfire = sorted(
        c
        for c in produced
        if c.startswith(
            "landfire_"
        )
        and c not in V5_FEATURE_SET
    )

    if unexpected_landfire:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"unexpected LANDFIRE fields: "
            f"{unexpected_landfire[:20]}"
        )

    directional_cols = list(
        DIRECTIONAL_V5_FEATURES
    )

    unreliable = ~features[
        "landfire_directional_context_valid"
    ].fillna(
        False
    )

    if (
        len(
            directional_cols
        )
        and features.loc[
            unreliable,
            directional_cols,
        ].notna().any(
            axis=1
        ).any()
    ):
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            "unreliable direction received directional features"
        )

    # Deterministic exact production order.
    features = features[
        EXPECTED_SHARD_COLUMNS
    ].copy()

    if len(
        features
    ) != expected_units:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            "feature row count mismatch"
        )

    if features[
        "unit_uid"
    ].nunique() != expected_units:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            "unit_uid uniqueness failed"
        )

    shard = shard_path_for(
        version_key,
        block_id,
    )

    tmp = shard.with_suffix(
        shard.suffix
        + ".tmp"
    )

    tmp.unlink(
        missing_ok=True
    )

    features.to_parquet(
        tmp,
        index=False,
    )

    tmp_check = validate_existing_shard(
        tmp,
        expected_units,
    )

    if not tmp_check[
        "valid"
    ]:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"temp shard validation failed: {tmp_check}"
        )

    tmp.replace(
        shard
    )

    final_check = validate_existing_shard(
        shard,
        expected_units,
    )

    if not final_check[
        "valid"
    ]:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"final shard validation failed: {final_check}"
        )

    return {
        "version_key": version_key,
        "block_id": block_id,
        "units": expected_units,
        "line_samples": len(
            line_samples
        ),
        "context_requests": len(
            requests_df
        ),
        "feature_rows": len(
            features
        ),
        "feature_columns": len(
            features.columns
        ),
        "directional_valid_units": (
            directional_valid_units
        ),
        "zero_direction_block": (
            zero_direction_block
        ),
        "structurally_filled_directional_features": len(
            structurally_filled_directional
        ),
        "shard_path": str(
            shard
        ),
    }

## 18. Audit existing production shards before resume

In [ ]:
existing_rows = []

for _, row in block_workload.iterrows():

    shard = shard_path_for(
        row[
            "landfire_version_key"
        ],
        row[
            "terrain_block_id"
        ],
    )

    check = validate_existing_shard(
        shard,
        int(
            row[
                "units"
            ]
        ),
    )

    existing_rows.append({
        "version_key": row[
            "landfire_version_key"
        ],
        "block_id": row[
            "terrain_block_id"
        ],
        "expected_units": int(
            row[
                "units"
            ]
        ),
        "shard_path": str(
            shard
        ),
        **check,
    })


existing_shard_audit = pd.DataFrame(
    existing_rows
)

display(
    existing_shard_audit[
        "valid"
    ].value_counts(
        dropna=False
    )
)

display(
    existing_shard_audit[
        "reason"
    ].value_counts(
        dropna=False
    )
)

existing_shard_audit.to_csv(
    OUTPUT_ROOT /
    "EXISTING_SHARD_AUDIT_BEFORE_RUN_V1.csv",
    index=False,
)

print(
    "VALID EXISTING:",
    int(
        existing_shard_audit[
            "valid"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_BLOCKS,
)

## 19. Full 690-block production loop

In [ ]:
run_rows = []
newly_processed = 0
run_start = time.time()

for block_number, (_, row) in enumerate(
    block_workload.iterrows(),
    start=1,
):

    version_key = row[
        "landfire_version_key"
    ]

    block_id = row[
        "terrain_block_id"
    ]

    expected_units = int(
        row[
            "units"
        ]
    )

    shard = shard_path_for(
        version_key,
        block_id,
    )

    precheck = validate_existing_shard(
        shard,
        expected_units,
    )

    if (
        RESUME_VALID_SHARDS
        and precheck[
            "valid"
        ]
    ):

        print(
            f"[{block_number:03d}/{EXPECTED_TOTAL_BLOCKS}] "
            f"SKIP {version_key} {block_id} "
            f"({expected_units:,} units)"
        )

        run_rows.append({
            "block_number": block_number,
            "version_key": version_key,
            "block_id": block_id,
            "units": expected_units,
            "status": "skipped_valid_existing",
            "elapsed_s": 0.0,
            "shard_path": str(
                shard
            ),
            "error": None,
        })

        continue

    print()
    print("=" * 100)
    print(
        f"[{block_number:03d}/{EXPECTED_TOTAL_BLOCKS}] "
        f"PROCESS {version_key} {block_id} "
        f"({expected_units:,} units)"
    )
    print("=" * 100)

    t0 = time.time()

    try:

        result = run_production_block(
            row
        )

        elapsed_s = (
            time.time()
            - t0
        )

        run_rows.append({
            "block_number": block_number,
            "version_key": version_key,
            "block_id": block_id,
            "units": expected_units,
            "status": "processed",
            "elapsed_s": elapsed_s,
            "directional_valid_units": result[
                "directional_valid_units"
            ],
            "zero_direction_block": result[
                "zero_direction_block"
            ],
            "structurally_filled_directional_features": result[
                "structurally_filled_directional_features"
            ],
            "shard_path": result[
                "shard_path"
            ],
            "error": None,
        })

        newly_processed += 1

        print(
            f"PASS in {elapsed_s/60:.2f} min"
        )

    except Exception as exc:

        elapsed_s = (
            time.time()
            - t0
        )

        run_rows.append({
            "block_number": block_number,
            "version_key": version_key,
            "block_id": block_id,
            "units": expected_units,
            "status": "FAILED",
            "elapsed_s": elapsed_s,
            "shard_path": str(
                shard
            ),
            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
        })

        pd.DataFrame(
            run_rows
        ).to_csv(
            PROGRESS_ROOT /
            "PRODUCTION_PROGRESS_LATEST_V1.csv",
            index=False,
        )

        print(
            traceback.format_exc()
        )

        raise

    if (
        newly_processed
        % PROGRESS_SAVE_EVERY_BLOCKS
        == 0
    ):

        pd.DataFrame(
            run_rows
        ).to_csv(
            PROGRESS_ROOT /
            "PRODUCTION_PROGRESS_LATEST_V1.csv",
            index=False,
        )

    if (
        newly_processed
        % GC_EVERY_BLOCKS
        == 0
    ):
        gc.collect()


production_progress = pd.DataFrame(
    run_rows
)

production_progress.to_csv(
    PROGRESS_ROOT /
    "PRODUCTION_PROGRESS_FINAL_V1.csv",
    index=False,
)

total_elapsed_s = (
    time.time()
    - run_start
)

print()
print("=" * 100)
print("FULL PRODUCTION LOOP COMPLETE")
print(
    f"Wall time: {total_elapsed_s/3600:.2f} h"
)
print("=" * 100)

## 20. Final 690-shard exact validation

In [ ]:
final_rows = []

for _, row in block_workload.iterrows():

    shard = shard_path_for(
        row[
            "landfire_version_key"
        ],
        row[
            "terrain_block_id"
        ],
    )

    check = validate_existing_shard(
        shard,
        int(
            row[
                "units"
            ]
        ),
    )

    final_rows.append({
        "version_key": row[
            "landfire_version_key"
        ],
        "block_id": row[
            "terrain_block_id"
        ],
        "expected_units": int(
            row[
                "units"
            ]
        ),
        "shard_path": str(
            shard
        ),
        **check,
    })


final_shard_audit = pd.DataFrame(
    final_rows
)

display(
    final_shard_audit[
        "valid"
    ].value_counts(
        dropna=False
    )
)

if not final_shard_audit[
    "valid"
].all():

    display(
        final_shard_audit[
            ~final_shard_audit[
                "valid"
            ]
        ]
    )

    raise RuntimeError(
        "One or more final shards are invalid."
    )

assert len(
    final_shard_audit
) == EXPECTED_TOTAL_BLOCKS

assert int(
    final_shard_audit[
        "rows"
    ].sum()
) == EXPECTED_TOTAL_UNITS

final_shard_audit.to_csv(
    OUTPUT_ROOT /
    "FINAL_SHARD_AUDIT_V1.csv",
    index=False,
)

print(
    "ALL 690 SHARDS: VALID"
)

print(
    "ROWS:",
    int(
        final_shard_audit[
            "rows"
        ].sum()
    ),
)

## 21. Exact global unit coverage

In [ ]:
uid_frames = []

for shard_path in final_shard_audit[
    "shard_path"
]:

    uid_frames.append(
        pd.read_parquet(
            Path(
                shard_path
            ),
            columns=[
                "unit_uid",
                "landfire_version_key",
            ],
        )
    )


produced_units = pd.concat(
    uid_frames,
    ignore_index=True,
)

expected_units_df = archived_base[
    [
        "unit_uid",
        "landfire_version_key",
    ]
].drop_duplicates().copy()

expected_set = set(
    expected_units_df[
        "unit_uid"
    ]
)

produced_set = set(
    produced_units[
        "unit_uid"
    ]
)

missing_units = sorted(
    expected_set
    - produced_set
)

unexpected_units = sorted(
    produced_set
    - expected_set
)

duplicate_rows = (
    len(
        produced_units
    )
    - produced_units[
        "unit_uid"
    ].nunique()
)

unit_coverage = pd.DataFrame({
    "metric": [
        "expected_units",
        "produced_rows",
        "produced_unique_units",
        "duplicate_rows",
        "missing_expected_units",
        "unexpected_units",
    ],
    "value": [
        len(
            expected_units_df
        ),
        len(
            produced_units
        ),
        produced_units[
            "unit_uid"
        ].nunique(),
        duplicate_rows,
        len(
            missing_units
        ),
        len(
            unexpected_units
        ),
    ],
})

display(unit_coverage)

assert len(
    expected_units_df
) == EXPECTED_TOTAL_UNITS

assert len(
    produced_units
) == EXPECTED_TOTAL_UNITS

assert produced_units[
    "unit_uid"
].nunique() == EXPECTED_TOTAL_UNITS

assert duplicate_rows == 0
assert not missing_units
assert not unexpected_units

version_unit_coverage = (
    produced_units.groupby(
        "landfire_version_key",
        as_index=False,
    )
    .agg(
        rows=(
            "unit_uid",
            "size",
        ),
        unique_units=(
            "unit_uid",
            "nunique",
        ),
    )
)

display(version_unit_coverage)

for version_key in ARCHIVED_VERSIONS:

    z = version_unit_coverage[
        version_unit_coverage[
            "landfire_version_key"
        ] == version_key
    ].iloc[0]

    assert int(
        z[
            "rows"
        ]
    ) == EXPECTED_VERSION_UNITS[
        version_key
    ]

    assert int(
        z[
            "unique_units"
        ]
    ) == EXPECTED_VERSION_UNITS[
        version_key
    ]

unit_coverage.to_csv(
    OUTPUT_ROOT /
    "GLOBAL_UNIT_COVERAGE_V1.csv",
    index=False,
)

version_unit_coverage.to_csv(
    OUTPUT_ROOT /
    "VERSION_UNIT_COVERAGE_V1.csv",
    index=False,
)

print(
    "GLOBAL EXACT UNIT COVERAGE: PASS"
)

## 22. Final production checks

In [ ]:
checks = [
    {
        "check": "exact_archived_version_set",
        "pass": set(
            archived_base[
                "landfire_version_key"
            ]
        ) == set(
            ARCHIVED_VERSIONS
        ),
    },
    {
        "check": "expected_total_blocks_690",
        "pass": len(
            block_workload
        ) == EXPECTED_TOTAL_BLOCKS,
    },
    {
        "check": "expected_total_units_132749",
        "pass": len(
            archived_base
        ) == EXPECTED_TOTAL_UNITS,
    },
    {
        "check": "all_10_archive_sources_validated",
        "pass": (
            len(
                archive_sources
            ) == 10
            and archive_sources[
                "acquisition_validated_bool"
            ].all()
        ),
    },
    {
        "check": "archive_source_metadata_contract_pass",
        "pass": bool(
            source_metadata_qa[
                metadata_check_cols
            ].all().all()
        ),
    },
    {
        "check": "archive_native_grid_phase_alignment_pass",
        "pass": bool(
            source_grid_alignment_qa[
                "native_grid_phase_aligned"
            ].all()
        ),
    },
    {
        "check": "all_690_expected_shards_valid",
        "pass": (
            len(
                final_shard_audit
            ) == EXPECTED_TOTAL_BLOCKS
            and final_shard_audit[
                "valid"
            ].all()
        ),
    },
    {
        "check": "all_shards_exact_ordered_612_column_schema",
        "pass": bool(
            final_shard_audit[
                "valid"
            ].all()
            and (
                final_shard_audit[
                    "columns"
                ] == 612
            ).all()
        ),
    },
    {
        "check": "all_shard_rows_sum_to_132749",
        "pass": int(
            final_shard_audit[
                "rows"
            ].sum()
        ) == EXPECTED_TOTAL_UNITS,
    },
    {
        "check": "all_expected_units_covered_exactly_once",
        "pass": (
            len(
                produced_units
            ) == EXPECTED_TOTAL_UNITS
            and produced_units[
                "unit_uid"
            ].nunique()
            == EXPECTED_TOTAL_UNITS
            and duplicate_rows == 0
            and len(
                missing_units
            ) == 0
            and len(
                unexpected_units
            ) == 0
        ),
    },
    {
        "check": "zero_direction_fill_restricted_to_frozen_directional_fields",
        "pass": bool(
            DIRECTIONAL_V5_FEATURE_SET.issubset(
                V5_FEATURE_SET
            )
            and len(
                DIRECTIONAL_V5_FEATURE_SET
            ) > 0
        ),
    },
    {
        "check": "archive_missing_value_contract_frozen",
        "pass": (
            EXPECTED_DECLARED_NODATA
            == 32767.0
            and ARCHIVE_EMBEDDED_NODATA_SENTINELS
            == (
                -9999.0,
            )
        ),
    },
]

checks_df = pd.DataFrame(
    checks
)

display(checks_df)

PRODUCTION_PASS = bool(
    checks_df[
        "pass"
    ].all()
)

print()
print(
    "04D1I ARCHIVED-VINTAGE FULL PRODUCTION:",
    (
        "PASS / READY FOR FIVE-VINTAGE CONSOLIDATION"
        if PRODUCTION_PASS
        else "REVIEW REQUIRED"
    ),
)

checks_df.to_csv(
    OUTPUT_ROOT /
    "FINAL_PRODUCTION_CHECKS_V1.csv",
    index=False,
)

## 23. Final report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(
            index=False
        )
    except Exception:
        return df.to_string(
            index=False
        )


run_summary = (
    production_progress.groupby(
        [
            "version_key",
            "status",
        ],
        as_index=False,
    )
    .agg(
        blocks=(
            "block_id",
            "size",
        ),
        units=(
            "units",
            "sum",
        ),
        elapsed_s=(
            "elapsed_s",
            "sum",
        ),
    )
)

report = f"""# 04D1I LANDFIRE Archived-Vintage Full Production

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Frozen workload

{tomd(version_workload)}

## Run summary

{tomd(run_summary)}

## Final shard validation

Valid shards: {int(final_shard_audit['valid'].sum())} / {len(final_shard_audit)}

Rows across shards: {int(final_shard_audit['rows'].sum())}

Columns per valid shard: 612

## Global unit coverage

{tomd(unit_coverage)}

## Version unit coverage

{tomd(version_unit_coverage)}

## Source contract

Declared archive TIFF NoData: `{EXPECTED_DECLARED_NODATA}`

Embedded archive sentinel(s): `{ARCHIVE_EMBEDDED_NODATA_SENTINELS}`

Analysis missing representation: `NaN`

## Final checks

{tomd(checks_df)}

## Decision

`{"PASS / READY FOR FIVE-VINTAGE CONSOLIDATION" if PRODUCTION_PASS else "REVIEW REQUIRED"}`
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "LANDFIRE_ARCHIVED_VINTAGE_FULL_PRODUCTION_REPORT_V1.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print("Saved:", REPORT_PATH)

## 24. Output inventory

In [ ]:
print("=" * 100)
print("04D1I OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "Feature shards:",
    len(
        list(
            SHARD_ROOT.glob(
                "*.parquet"
            )
        )
    ),
)

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)